# 하남교산 격자별 ECR 최종 산출

하남교산 격자 단위로 ECR(위험지수)를 산출합니다.

- 사고/EPDO: `13._교통사고이력.geojson` + `02._격자_(하남교산).geojson` point-in-polygon
- 7개 인자: `08_격자_종합위험지수.csv` (하남 격자만)
- ECR: `entropy_composite_risk = EPDO × (1 + correction_index)`
- correction_index: `09_엔트로피_가중치.csv` 기반, min-max 정규화

**출력:** `epdo_analysis/output/13_하남교산_격자별_ECR_최종.csv`

In [ ]:
import csv
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

## 경로 및 상수 설정

In [ ]:
BASE_DIR   = Path(".").resolve()
DATA_DIR   = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "epdo_analysis" / "output"

# 입력 파일
INPUT_GRID    = DATA_DIR   / "02._격자_(하남교산).geojson"
INPUT_ACC     = DATA_DIR   / "13._교통사고이력.geojson"
INPUT_RISK    = OUTPUT_DIR / "08_격자_종합위험지수.csv"
INPUT_INFRA   = OUTPUT_DIR / "00_격자별_통합데이터.csv"
INPUT_ENTROPY = OUTPUT_DIR / "09_엔트로피_가중치.csv"

# 출력 파일
OUT_PATH = OUTPUT_DIR / "13_하남교산_격자별_ECR_최종.csv"

# 경로 확인
for name, p in [
    ("하남격자",       INPUT_GRID),
    ("교통사고",       INPUT_ACC),
    ("종합위험지수",   INPUT_RISK),
    ("통합데이터",     INPUT_INFRA),
    ("엔트로피가중치", INPUT_ENTROPY),
]:
    print(f"{name:10s}: {'OK' if p.exists() else 'NOT FOUND'}  {p}")

In [ ]:
# EPDO 가중치
EPDO_WEIGHTS = {"사망": 391, "중상": 69, "경상": 8, "부상신고": 6, "상해없음": 1, "기타불명": 8}

# 7개 위험 인자
FACTOR_COLS = [
    "elderly_res_ratio", "vuln_peak_pop", "grid_congestion",
    "grid_avg_speed", "weekend_ratio", "gap_cnt", "vuln_float_ratio",
]

# 안전시설 컬럼
SAFETY_COLS = [
    "crosswalk_cnt", "child_zone_cnt", "speedbump_cnt",
    "cctv_cnt", "cctv_cam_cnt", "bus_stop_cnt",
]

## 헬퍼 함수

In [ ]:
def load_entropy_weights(path: Path) -> dict:
    w = {}
    with open(path, "r", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            w[row["인자"]] = float(row["가중치"])
    return w


def accident_epdo_point_in_polygon(grid_gdf: gpd.GeoDataFrame, acc_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """13번 교통사고이력 + 02번 격자 point-in-polygon"""
    acc_gdf = acc_gdf.to_crs(grid_gdf.crs)
    result = []
    for _, row in grid_gdf.iterrows():
        gid  = row["gid"]
        poly = row.geometry
        pts  = acc_gdf[acc_gdf.intersects(poly)]
        cnt  = {k: 0 for k in EPDO_WEIGHTS}
        epdo = 0
        for _, p in pts.iterrows():
            sv = (p.get("injury_svrity") or "").strip()
            if   "사망"   in sv: cnt["사망"]   += 1; epdo += EPDO_WEIGHTS["사망"]
            elif "중상"   in sv: cnt["중상"]   += 1; epdo += EPDO_WEIGHTS["중상"]
            elif "경상"   in sv: cnt["경상"]   += 1; epdo += EPDO_WEIGHTS["경상"]
            elif "부상신고" in sv: cnt["부상신고"] += 1; epdo += EPDO_WEIGHTS["부상신고"]
            elif "상해없음" in sv or sv == "": cnt["상해없음"] += 1; epdo += EPDO_WEIGHTS["상해없음"]
            else: cnt["기타불명"] += 1; epdo += EPDO_WEIGHTS["기타불명"]
        result.append({
            "grid_gid": gid,
            "사고건수": len(pts), **{k: cnt[k] for k in cnt}, "EPDO점수": epdo,
        })
    return pd.DataFrame(result)


def calc_gap_from_infra(infra_df: pd.DataFrame, safety_cols: list) -> pd.DataFrame:
    """00_격자별_통합데이터 기반 gap_cnt, gap_items 산출"""
    means = {c: infra_df[c].fillna(0).replace("", 0).astype(float).mean() for c in safety_cols}
    def _gap(r):
        gaps = []
        for c in safety_cols:
            v = float(r.get(c, 0) or 0)
            if v == 0:
                gaps.append(f"{c}(없음)")
            elif means[c] > 0 and v < means[c] * 0.5:
                gaps.append(f"{c}(부족)")
        return len(gaps), " | ".join(gaps) if gaps else "-"
    infra_df = infra_df.copy()
    res = infra_df.apply(_gap, axis=1)
    infra_df["gap_cnt"]   = [r[0] for r in res]
    infra_df["gap_items"] = [r[1] for r in res]
    return infra_df

## Step 1 · 하남교산 격자 로드

In [ ]:
grid_gdf = gpd.read_file(INPUT_GRID)
grid_gdf["gid"] = grid_gdf["gid"].astype(str)
hanam_gids = set(grid_gdf["gid"])

print(f"하남교산 격자 수: {len(grid_gdf):,}")

## Step 2 · 교통사고 Point-in-Polygon

In [ ]:
acc_gdf = gpd.read_file(INPUT_ACC)
acc_agg = accident_epdo_point_in_polygon(grid_gdf, acc_gdf)
acc_agg = acc_agg[acc_agg["사고건수"] > 0]

print(f"사고 발생 격자 수: {len(acc_agg):,}")
acc_agg.head()

## Step 3 · 종합위험지수에서 7개 인자 + gap_items 추출

In [ ]:
risk       = pd.read_csv(INPUT_RISK, encoding="utf-8-sig")
risk_hanam = risk[risk["grid_gid"].astype(str).isin(hanam_gids)].copy()

print(f"하남 해당 위험지수 행: {len(risk_hanam):,}")

## Step 4 · 통합데이터로 gap 보완

In [ ]:
infra       = pd.read_csv(INPUT_INFRA, encoding="utf-8-sig")
infra_hanam = infra[infra["grid_gid"].astype(str).isin(hanam_gids)].copy()
infra_hanam = calc_gap_from_infra(infra_hanam, SAFETY_COLS)

merged = acc_agg.merge(
    risk_hanam[["grid_gid"] + FACTOR_COLS + ["gap_items"]],
    on="grid_gid", how="left"
)
merged = merged.merge(
    infra_hanam[["grid_gid", "gap_cnt", "gap_items"]],
    on="grid_gid", how="left", suffixes=("", "_infra")
)
merged["gap_cnt"]        = merged["gap_cnt"].fillna(merged["gap_cnt_infra"]).astype(int)
merged["gap_items"]      = merged["gap_items"].fillna(merged["gap_items_infra"])
merged["핵심_인프라공백"] = merged["gap_items"]

for c in FACTOR_COLS:
    if c in merged.columns and c != "gap_cnt":
        merged[c] = pd.to_numeric(merged[c], errors="coerce").fillna(0)
merged["grid_avg_speed"]  = pd.to_numeric(merged["grid_avg_speed"],  errors="coerce").fillna(0)
merged["grid_congestion"] = pd.to_numeric(merged["grid_congestion"], errors="coerce").fillna(0)

print(f"병합 결과 행 수: {len(merged):,}")

## Step 5 · ECR(위험지수) 산출

In [ ]:
w = load_entropy_weights(INPUT_ENTROPY)

# 인자 min-max 정규화
for col in FACTOR_COLS:
    s  = pd.to_numeric(merged[col], errors="coerce").fillna(0)
    mn, mx = s.min(), s.max()
    merged[f"_n_{col}"] = (s - mn) / (mx - mn) if (mx - mn) > 0 else 0.5

merged["correction_index"]      = sum(w.get(c, 0) * merged[f"_n_{c}"] for c in FACTOR_COLS if c in w)
merged["entropy_composite_risk"] = (merged["EPDO점수"] * (1 + merged["correction_index"])).round(2)
merged["correction_index"]       = merged["correction_index"].round(6)

merged["위험등급"] = merged["entropy_composite_risk"].apply(
    lambda x: "고위험" if x >= 111.12 else ("중위험" if x >= 77.38 else "저위험")
)

print(merged[["grid_gid", "EPDO점수", "correction_index", "entropy_composite_risk", "위험등급"]].head())

## 결과 저장

In [ ]:
out_cols = [
    "grid_gid", "사고건수", "사망", "중상", "경상", "부상신고", "상해없음", "EPDO점수",
    "elderly_res_ratio", "vuln_float_ratio", "vuln_peak_pop", "weekend_ratio",
    "grid_avg_speed", "grid_congestion", "gap_cnt", "gap_items",
    "correction_index", "entropy_composite_risk", "위험등급", "핵심_인프라공백",
]
out = merged[[c for c in out_cols if c in merged.columns]].copy()
out = out.sort_values("entropy_composite_risk", ascending=False)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"저장: {OUT_PATH}")
print(f"격자 수: {len(out):,}개")
print("\nTop 10 (위험지수 기준):")
for _, r in out.head(10).iterrows():
    print(f"  {r['grid_gid']:12s}  위험지수={r['entropy_composite_risk']:>7.2f}")